# bunq: making suspicious-activity reporting to France's FIU fast, complete and measurable

One notebook, four parts: **(a)** public context (bunq, Tracfin, ECB) · **(b)** from transactions to alerts on IBM AML (public, synthetic) ·
**(c)** the reporting desk (**SIMULATED**): the eight hidden events, found from the data · **(d)** the pre-fill effect by difference-in-differences.

Inputs come from `src/run_sql.py` (DuckDB) and `src/findings.py`. Everything about the desk is simulated and is **not** bunq's data, process or performance.

In [1]:
import json, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import duckdb, numpy as np, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC, SIM = ROOT / "data" / "processed", ROOT / "data" / "simulated"
pq = lambda n: pd.read_parquet(PROC / f"{n}.parquet")
F = json.loads((PROC / "findings.json").read_text(encoding="utf-8"))
con = duckdb.connect()
for p in SIM.glob("*.parquet"):
    con.execute(f"CREATE VIEW {p.stem} AS SELECT * FROM read_parquet('{p.as_posix()}')")
q = lambda s: con.execute(s).df()
pd.set_option("display.width", 140)

## (a) Public context
### bunq, as stated in its 2025 annual report (page numbers in the table)

In [2]:
pd.read_csv(PROC / "bunq_facts.csv")[["fact", "value_2025", "value_2024", "unit", "page"]]

,fact,value_2025,value_2024,unit,page
0,users,22.0,NaN,million users,"6, 25, 55"
1,fte_total,665.0,650.0,FTE at year end,79
2,fte_onboarding_support_compliance,280.0,229.0,FTE at year end,79
3,fee_income_france_belgium,3256.0,3596.0,EUR thousand,78
4,fee_income_total,51829.0,52101.0,EUR thousand,78
5,aml_ai,NaN,NaN,statement,21
6,supervisor,NaN,NaN,statement,52


### Tracfin 2025 (read in a browser) and ECB payment fraud, France vs Netherlands

In [3]:
t = pd.read_csv(PROC / "tracfin_2025.csv")[["metric", "value", "unit"]]
display(t)
print("ECB unit multiplier:", F["ecb"]["unit_mult"], "-> values in millions")
print("Card fraud, bp of value, 2025-S2:", {k: round(v, 2) for k, v in F["ecb"]["card_fraud_rate_bp_2025S2"].items()})
print(f"FR credit-transfer fraud 2025-S2: EUR {F['ecb']['fr_ct_fraud_2025S2_m']:.1f}m ({F['ecb']['fr_ct_fraud_growth']:+.0%} vs 2024-S2)")

,metric,value,unit
0,str_total_2025,278484.00,declarations
1,str_growth_2025,0.32,share
2,str_multiple_since_2015,6.00,times
3,financial_sector_str,258470.00,declarations
4,financial_sector_share,0.93,share
5,financial_sector_growth,0.31,share
6,banks_growth,0.45,share
7,casp_str,4850.00,declarations
8,non_financial_share,0.07,share
9,non_financial_growth,0.38,share


ECB unit multiplier: 6 -> values in millions
Card fraud, bp of value, 2025-S2: {'FR': 4.74, 'NL': 2.05}
FR credit-transfer fraud 2025-S2: EUR 249.4m (+28% vs 2024-S2)


## (b) From transactions to alerts (IBM AML HI-Small, synthetic, CDLA-Sharing-1.0)
Load checks first: a row-count guard caught a join bug in my own pipeline (see `docs/DATA_ISSUES_LOG.md`).

In [4]:
display(pd.read_csv(PROC / "ibm_load_checks.csv").T)
pq("ibm_rule_results")[["rule", "flagged_accounts", "flagged_positives", "precision", "recall", "alerts_per_1000_accounts"]].round(4)

,0
rows_in_file,5078345
rows_after_exact_dedup,5078336
exact_duplicates_removed,9
laundering_rows,5177
accounts_seen,515088
first_day,2022-09-01
last_day,2022-09-18
pattern_rows,3209
laundering_attempts,370


,rule,flagged_accounts,flagged_positives,precision,recall,alerts_per_1000_accounts
0,Any rule (R1-R4),18446,842,0.0456,0.1325,35.8114
1,R1 fan-out (8+ payees / week),5683,228,0.0401,0.0359,11.0331
2,R2 fan-in (8+ payers / week),3687,348,0.0944,0.0547,7.1580
3,R3 structuring (3+ payments USD 8.5-10k / week),5140,128,0.0249,0.0201,9.9789
4,R4 rapid pass-through (80-100% out same day),5046,241,0.0478,0.0379,9.7964


In [5]:
i = F["ibm"]
print(f"Base rate {i['base_rate']:.2%} | any-rule precision {i['any_precision']:.2%} = {i['lift']:.1f}x the base rate | recall {i['any_recall']:.1%}")
print(f"At 20 min per alert (assumption): {i['analyst_hours_per_true_positive']:.1f} analyst hours per true positive")
display(pq("ibm_fanout_curve").round(4).iloc[::3])
pq("ibm_typology_coverage").round(3)

Base rate 1.23% | any-rule precision 4.56% = 3.7x the base rate | recall 13.2%
At 20 min per alert (assumption): 7.3 analyst hours per true positive


,threshold,flagged_accounts,flagged_positives,precision,recall
0,3,52638,1230,0.0234,0.1935
3,6,15722,381,0.0242,0.0599
6,9,3571,168,0.0470,0.0264
9,12,1047,91,0.0869,0.0143
12,15,336,36,0.1071,0.0057
15,18,124,18,0.1452,0.0028


,typology,attempts,detected_attempts,detection_rate
0,CYCLE,54,40.0,0.741
1,FAN-OUT,48,37.0,0.771
2,SCATTER-GATHER,44,35.0,0.795
3,BIPARTITE,49,41.0,0.837
4,RANDOM,41,36.0,0.878
5,STACK,43,38.0,0.884
6,FAN-IN,40,36.0,0.900
7,GATHER-SCATTER,51,47.0,0.922


## (c) The reporting desk (SIMULATED): eight hidden events, found from the data
The answer key (`data/_hidden_truth.csv`) is **not** read here. Each event is found with a query, then explained.

### Event 1 — a new monitoring rule (real operational effect)

In [6]:
m = q('''SELECT date_trunc('month', alert_ts_utc) AS month, desk, count(*) FILTER (WHERE source='monitoring_rule') AS rule_alerts
         FROM alerts GROUP BY 1, 2 ORDER BY 1, 2''').pivot(index="month", columns="desk", values="rule_alerts")
m["FR/BNL"] = (m.FR / m.BNL).round(2)
display(m.T)
t = pq("sim_timeliness"); fr = t[t.desk == "FR"].set_index("month")
fr[["reports", "true_median_days", "on_time_rate"]].round(2).T

month,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,2026-08-01,2026-09-01,2026-10-01
desk,,,,,,,,,,,,
BNL,883.00,1345.00,978.00,825.0,917.00,936.00,900.00,887.00,951.00,894.00,916.00,927.00
FR,1157.00,1823.00,1363.00,1156.0,2322.00,2293.00,2345.00,2198.00,2235.00,2306.00,2265.00,2282.00
FR/BNL,1.31,1.36,1.39,1.4,2.53,2.45,2.61,2.48,2.35,2.58,2.47,2.46


month,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,2026-08-01,2026-09-01,2026-10-01
reports,122.00,246.00,240.00,213.00,269.00,230.00,408.00,460.00,338.00,361.00,348.00,312.00
true_median_days,10.93,12.43,15.13,11.78,13.10,20.28,24.18,14.52,11.83,11.05,11.00,10.81
on_time_rate,0.92,0.83,0.42,0.78,0.73,0.03,0.05,0.52,0.85,0.88,0.91,0.94


France's rule alerts jump from March while Benelux stays flat (the FR/BNL ratio jumps in March): a France-only change.
The triage backlog then grows and time to file doubles in April–May, until capacity catches up in June–July. **Real effect** → capacity and rule tuning, a CAP item is justified.

### Event 6 — the December spike (real inflow, not a failure)

In [7]:
q('''SELECT desk, date_trunc('week', alert_ts_utc)::DATE AS week, count(*) AS alerts,
            avg((typology_hint = 'APP scam (year-end)')::INT) AS app_scam_share, avg((case_id IS NOT NULL)::INT) AS escalation_rate
     FROM alerts WHERE alert_ts_utc BETWEEN '2025-11-17' AND '2026-01-25' GROUP BY 1, 2 ORDER BY 1, 2''').pivot(index="week", columns="desk")

alerts      app_scam_share           escalation_rate          
desk          BNL   FR            BNL        FR             BNL        FR
week                                                                     
2025-11-17    286  361       0.000000  0.000000        0.307692  0.315789
2025-11-24    253  329       0.000000  0.000000        0.320158  0.343465
2025-12-01    278  414       0.000000  0.000000        0.287770  0.355072
2025-12-08    387  537       0.322997  0.283054        0.320413  0.331471
2025-12-15    389  514       0.298201  0.289883        0.339332  0.315175
2025-12-22    379  489       0.292876  0.316973        0.292876  0.325153
2025-12-29    355  559       0.253521  0.291592        0.326761  0.280859
2026-01-05    260  365       0.000000  0.000000        0.307692  0.336986
2026-01-12    251  333       0.000000  0.000000        0.358566  0.309309
2026-01-19    249  302       0.000000  0.000000        0.361446  0.331126

Both desks rise together from early December, a third of the extra alerts are tagged as year-end scams, and the escalation rate does not change:
a **seasonal inflow**. Plan capacity for it next year; do not treat it as a process failure.

### Event 2 — a schema change drops fields (data quality)

In [8]:
display(q('''SELECT desk, date_trunc('month', draft_ts_utc)::DATE AS month, missing_fields_at_draft AS missing, count(DISTINCT report_id) AS n
             FROM reports WHERE missing_fields_at_draft <> '' GROUP BY 1, 2, 3 ORDER BY 1, 2''').pivot_table(index=["desk", "missing"], columns="month", values="n", fill_value=0))
qm = pq("sim_quality_monthly"); qm[qm.desk == "FR"].set_index("month")[["complete_at_draft", "complete_at_submission", "qa_return_rate"]].round(3).T

month                                        2025-11-01  2025-12-01  2026-01-01  2026-02-01  2026-03-01  2026-04-01  2026-05-01  \
desk missing                                                                                                                      
BNL  narrative                                      3.0         3.0         1.0         1.0         0.0         1.0         1.0   
FR   counterparty_country;counterparty_iban         0.0         0.0         0.0         0.0         0.0         0.0        44.0   
     narrative                                      0.0         3.0         1.0         0.0         2.0         0.0         4.0   

month                                        2026-06-01  2026-07-01  2026-08-01  2026-09-01  2026-10-01  
desk missing                                                                                             
BNL  narrative                                      1.0         2.0         2.0         1.0         3.0  
FR   counterparty_country;counterparty_iban        65.0        19.0         0.0         0.0         0.0  
     narrative                                      5.0         1.0         3.0         5.0         1.0

month,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,2026-08-01,2026-09-01,2026-10-01
complete_at_draft,1.000,0.988,0.996,1.000,0.993,1.000,0.909,0.833,0.929,0.992,0.986,0.997
complete_at_submission,1.000,0.996,1.000,1.000,0.996,1.000,0.980,0.959,0.982,1.000,0.997,1.000
qa_return_rate,0.025,0.041,0.033,0.028,0.037,0.017,0.098,0.165,0.083,0.053,0.046,0.048


Only France, only **counterparty country and IBAN**, only from mid-May to mid-July: the signature of a schema/mapping change, not of analysts.
QA catches most drafts (returns jump), but ~4% of June reports still went out incomplete. Fix: a **blocking completeness check** (`src/prefill_report.py`).

### Event 3 — duplicate submissions (retry bug)

In [9]:
display(pq("sim_duplicates"))
q('''SELECT count(*) AS submission_rows, count(DISTINCT report_id) AS reports FROM reports''')

,desk,month,duplicate_rows,median_minutes_after_first
0,BNL,2026-06-01,6,6.919287
1,BNL,2026-07-01,4,8.257529
2,FR,2026-06-01,15,7.976916
3,FR,2026-07-01,11,8.911945


,submission_rows,reports
0,5439,5403


Same `report_id`, new `submission_id`, 2–15 minutes later, only in June–July, both desks: a **retry bug**. Counting rows inflates the number of reports; count `report_id` (window function in `sql/04_quality.sql`).

### Event 4 — the 40% improvement that is not one

In [10]:
fr[["tool_median_days", "true_median_days", "case_open_to_submit_median_days", "tool_clock_start"]].round(2).tail(5)

,tool_median_days,true_median_days,case_open_to_submit_median_days,tool_clock_start
month,,,,
2026-06-01,14.52,14.52,7.43,alert_ts
2026-07-01,11.83,11.83,7.42,alert_ts
2026-08-01,11.05,11.05,6.59,alert_ts
2026-09-01,6.68,11.00,6.68,case_opened_ts
2026-10-01,6.37,10.81,6.37,case_opened_ts


The tool's own field falls by about 40% in September, but the median recomputed **from the alert** is flat, and the tool's figure now equals
the *case-opening → submission* time: the tool's **clock start moved** (`tool_clock_start`). **Not credited.** Rule: recompute KPIs from raw timestamps
and keep one definition (`docs/KPI_DEFINITIONS.md`).

### Event 8 — UTC timestamps in a Paris deadline check

In [11]:
r = q('''WITH r AS (SELECT report_id, min(submitted_ts_utc) AS sub FROM reports GROUP BY 1)
          SELECT hour((r.sub AT TIME ZONE 'UTC') AT TIME ZONE 'Europe/Paris') AS paris_hour, count(*) AS n FROM r GROUP BY 1 ORDER BY 1''')
print("Submissions by Paris hour (late-night rush visible at 20-01h):", dict(zip(r.paris_hour, r.n)))
print(f"Reports whose on-time status changes when the deadline is checked on Paris dates: {F['sim']['utc_misclassified']} ({F['sim']['utc_misclassified_share']:.1%})")

Submissions by Paris hour (late-night rush visible at 20-01h): {0: 60, 1: 38, 9: 529, 10: 594, 11: 566, 12: 592, 13: 569, 14: 557, 15: 561, 16: 595, 17: 523, 20: 26, 21: 65, 22: 58, 23: 70}
Reports whose on-time status changes when the deadline is checked on Paris dates: 117 (2.2%)


Deadline-day submissions after midnight in Paris are still "today" in UTC, so a UTC check marks late reports as on time (~2%). Fix and tests:
`on_time()` in `src/prefill_report.py`, `tests/test_prefill.py`.

### Event 7 — CAP items

In [12]:
display(pq("sim_cap_by_team"))
s = pq("sim_cap_status"); s[s.cap_state.isin(["overdue", "closed without evidence"])][["cap_id", "item", "owner_team", "due_date", "closed_date", "cap_state"]]

,owner_team,items,overdue,closed_without_evidence,closed_on_time_with_evidence
0,T3,3,2.0,0.0,0.0
1,T1,4,0.0,0.0,4.0
2,T2,6,0.0,0.0,6.0
3,T4,11,0.0,0.0,11.0
4,T5,3,0.0,1.0,2.0


,cap_id,item,owner_team,due_date,closed_date,cap_state
23,cap_005,training refresh,T5,2026-06-14,2026-06-11,closed without evidence
25,cap_025,four-eyes review,T3,2026-08-31,NaT,overdue
26,cap_026,FIU feedback loop,T3,2026-09-30,NaT,overdue


Both overdue items sit in **team T3**; one item was closed **without evidence** and counts as not done. Weekly CAP review by owner team; evidence required to close.

## (d) Event 5 — did the August pre-fill work? Difference-in-differences

In [13]:
display(pq("sim_did_cells").round(3))
d = F["did"]
print(f"France before/after (naive): {d['naive_fr_before_after_pct']:+.1%}")
print(f"Difference-in-differences:   {d['did_pct']:+.1%}  95% CI [{d['ci_low_pct']:+.1%}, {d['ci_high_pct']:+.1%}]  (bootstrap over weeks, {d['n_boot']} draws)")
w = pq("sim_weekly_effort").pivot(index="week", columns="desk", values="geo_mean_hours")
pre = w.loc[:"2026-08-02"]
print("Pre-period parallel-trends check: slope of FR-BNL gap per week =", round(np.polyfit(np.arange(len(pre)), (pre.FR - pre.BNL).values, 1)[0], 4), "hours")

,desk,period,cases,mean_log_effort,geo_mean_hours
0,BNL,post,1128,1.750,5.757
1,BNL,pre,1149,1.734,5.662
2,FR,post,2502,1.370,3.934
3,FR,pre,2925,1.611,5.006


France before/after (naive): -21.4%
Difference-in-differences:   -22.7%  95% CI [-26.9%, -18.4%]  (bootstrap over weeks, 2000 draws)
Pre-period parallel-trends check: slope of FR-BNL gap per week = -0.0109 hours


Outcome = log analyst effort per case (so the effect reads as a % change). Benelux, which did not get the pre-fill, is the control:
it absorbs the September shift in case mix that both desks share. The estimate (about −23%, interval covering the true −25%) is the effect to report;
the September "−40%" on time to file is not.

## Conclusions
1. **Quality before volume** — Tracfin's own 2025 message. Block incomplete drafts; monitor completeness at draft.
2. **One definition per KPI** — clock from the alert, Paris dates, one report per `report_id`.
3. **Explain every step change** before crediting it: real (new rule, December), data (schema, duplicates, UTC) or definition (clock).
4. **Measure the legal requirements** that matter for the AMLR (from 10 July 2027): pre-execution reports, follow-ups, 5-working-day FIU requests.
5. **CAP items** are closed only with evidence.

*Simulated desk: not bunq's data, processes or performance. Legal points: to be confirmed with Compliance / Legal.*